# CNN Challenge — Colab GPU driver

ITCS 6169/8169 Assignment 1. This notebook is a **thin driver only**: it clones the
repository, installs dependencies, prepares the data, and calls the same
`train.py` that runs locally. There is deliberately no training logic here, so the
numbers reported in the README come from the code that is actually in the repo.

Before running: **Runtime → Change runtime type → GPU (T4)**.

In [ ]:
# 0. Confirm we actually have a GPU before spending time on setup.
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

## 1. Clone the repository

The repository is private during the assignment, so cloning needs a GitHub
personal access token (Settings → Developer settings → Personal access tokens →
Fine-grained, with read access to this repo). `getpass` keeps the token out of the
notebook output and out of git history.

In [ ]:
import os
from getpass import getpass

GITHUB_USER = 'smritib4'
REPO_NAME = 'cnn-challenge-scene-classification'

if not os.path.isdir(REPO_NAME):
    token = getpass('GitHub token (leave blank if the repo is public): ').strip()
    auth = f'{GITHUB_USER}:{token}@' if token else ''
    !git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO_NAME}.git

%cd {REPO_NAME}
!git log --oneline -n 5

In [ ]:
# 2. Dependencies. Colab already ships a CUDA torch build, so torch/torchvision
# are skipped to avoid replacing it with a different (possibly CPU) wheel.
!pip install -q PyYAML scikit-learn pandas tqdm gdown
!python -c "import yaml, sklearn, pandas, torch, torchvision; print('deps OK')"

## 3. Data

Recommended path: download the dataset archive from the link in the assignment PDF
once, put it in your Google Drive, and mount Drive here. That is faster and far
more reliable than crawling the shared Drive folder file by file, and it survives
runtime restarts.

Set `DATASET_ZIP` to the archive's path in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

DATASET_ZIP = '/content/gdrive/MyDrive/ITCS_6169_8169/assignment1_dataset.zip'  # <-- edit

import os
assert os.path.isfile(DATASET_ZIP), f'Not found: {DATASET_ZIP}'
print('Found archive:', DATASET_ZIP, f'({os.path.getsize(DATASET_ZIP) / 1e6:.1f} MB)')

In [ ]:
# Unpack into data/train/<class>/ and data/test/<class>/, and verify the counts.
# This prints per-class totals and warns if the training set is not 2,400 images.
!python scripts/prepare_data.py --zip "{DATASET_ZIP}"

## 4. Sanity check the pipeline

Two epochs at 64px on synthetic data. Cheap insurance that nothing is broken
before committing to a long run.

In [ ]:
!python scripts/make_dummy_data.py --out data_dummy
!python train.py --config configs/smoke.yaml --data-root data_dummy --output-root runs_smoke

## 5. The experiment ladder

Each config changes approximately one factor from the previous one. Run them in
order; every run appends a row to `experiments/results/results.csv`.

**Do not pass `--evaluate-test` here.** The test set is scored once, in section 7,
for the single model already selected on validation accuracy.

In [ ]:
# Cheap rows first (minutes each) so problems surface early.
!python train.py --config configs/00_baseline_tnet.yaml
!python train.py --config configs/01_baseline_plus_color.yaml
!python train.py --config configs/03_resnet18_linear_probe.yaml

In [ ]:
!python train.py --config configs/02_smallcnn_scratch.yaml
!python train.py --config configs/04_resnet18_finetune.yaml
!python train.py --config configs/05_resnet50_finetune.yaml

In [ ]:
!python train.py --config configs/06_resnet50_strong_aug.yaml
!python train.py --config configs/07_resnet50_mixup.yaml
!python train.py --config configs/08_convnext_tiny.yaml

In [ ]:
# Review the ladder so far.
import pandas as pd
results = pd.read_csv('experiments/results/results.csv')
display(results[['experiment', 'model', 'img_size', 'augment', 'epochs', 'params', 'val_acc', 'train_time_s']])

## 6. Final model

`configs/best.yaml` is the recipe assembled from whatever the ladder showed to
work. Repeating it across seeds separates a real improvement from split noise —
with a ~480-image validation set, a 1% difference is about five images.

In [ ]:
!python train.py --config configs/best.yaml --seed 0
!python train.py --config configs/best.yaml --seed 1
!python train.py --config configs/best.yaml --seed 2

## 7. Test evaluation — run once

Pick the checkpoint with the best **validation** accuracy, then score it on the
test set a single time.

In [ ]:
CHECKPOINT = 'runs/best_seed0/best.pt'  # <-- set to the best-validation run

!python evaluate.py --checkpoint {CHECKPOINT} --split val
!python evaluate.py --checkpoint {CHECKPOINT} --split test \
    --report reports/final_test_report.json \
    --confusion-matrix reports/confusion_matrix.png

In [ ]:
from IPython.display import Image, display
display(Image('reports/confusion_matrix.png'))

## 8. Persist artifacts

Colab runtimes are ephemeral. Copy the checkpoint and result files to Drive, then
commit the small text artifacts (results CSV, reports, figures) back to the repo.
The checkpoint itself is too large for git and is linked from the README instead.

In [ ]:
import shutil, os, glob

DEST = '/content/gdrive/MyDrive/ITCS_6169_8169/assignment1_artifacts'
os.makedirs(DEST, exist_ok=True)

shutil.copy(CHECKPOINT, os.path.join(DEST, 'best.pt'))
for pattern in ('experiments/results/*.csv', 'reports/*', 'runs/*/summary.json', 'runs/*/history.jsonl'):
    for path in glob.glob(pattern):
        target = os.path.join(DEST, path.replace('/', '_'))
        shutil.copy(path, target)
print('Copied artifacts to', DEST)
print(os.listdir(DEST))

In [ ]:
# Push the text artifacts back so the repo records the actual numbers.
!git config user.name "Smriti Bhemireddy"
!git config user.email "smritibhemireddy@gmail.com"
!git add -f experiments/results/*.csv reports runs/*/summary.json runs/*/history.jsonl
!git -c core.hooksPath=/dev/null commit -m "Add experiment results and final evaluation artifacts from Colab T4 runs"
!git push origin main